<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module1/m2_notebook1_stellar_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔭 Module 2 · Notebook 1
# MLP for Stellar Spectra Classification

---

**Module 2 · Deep Neural Networks**  
**Estimated time:** 40 minutes  
**Level:** Beginner — assumes basic Python and familiarity with the Module 2 reading material

---

## 📋 What This Notebook Does

In this notebook you will build, train, and evaluate a **Multi-Layer Perceptron (MLP)** — a fully connected neural network — applied to a real scientific problem: classifying stellar objects from spectral data.

Stellar spectra tell us what a star (or galaxy, or quasar) is made of, how hot it is, how fast it is moving, and how far away it is. The Sloan Digital Sky Survey (SDSS) has catalogued millions of such objects. Your job is to train a neural network to automatically classify each object as a **Star**, **Galaxy**, or **Quasar (QSO)** based on its spectral features.

By the end of this notebook you will have:

- Loaded, explored, and preprocessed a simulated stellar spectra dataset
- Built a fully connected MLP with dropout regularisation in TensorFlow/Keras
- Trained the model with early stopping and plotted loss and accuracy curves
- Evaluated performance using a confusion matrix and per-class metrics
- Visualised the learned weight matrix from the first hidden layer

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Install and import libraries |
| **1. Load Data** | Load and explore SDSS stellar spectra |
| **2. Preprocess** | Normalise features, encode labels, split data |
| **3. Build MLP** | Define the network architecture with dropout |
| **4. Train** | Fit with early stopping, plot learning curves |
| **5. Evaluate** | Confusion matrix and classification report |
| **6. Interpret** | Visualise learned weights from the first layer |
| **7. Exercises** | Guided reflection and optional experiments |

---

> **Before you start:** Make sure you have read the Lesson 1 reading material on neural network structure, neurons, weights, biases, activation functions, and backpropagation. This notebook puts all of those concepts into practice.

---
## Step 0 — Setup: Install and Import Libraries

We will use:
- `numpy` and `pandas` for data handling
- `matplotlib` and `seaborn` for visualisation
- `scikit-learn` for preprocessing and evaluation metrics
- `tensorflow` / `keras` for building and training the neural network

All of these are pre-installed on Google Colab. Run the cell below to import them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

sns.set(style='whitegrid')
%matplotlib inline

print(f'TensorFlow version: {tf.__version__}')
print('✅ All libraries imported successfully.')

---
## Step 1 — Load and Explore the Data

### 1.1 — About the Dataset

We are working with a **simulated dataset** based on the structure of real SDSS (Sloan Digital Sky Survey) spectral observations. Each row represents one celestial object and contains:

| Feature | Description |
|---------|-------------|
| `u` | Ultraviolet filter magnitude |
| `g` | Green filter magnitude |
| `r` | Red filter magnitude |
| `i` | Near-infrared filter magnitude |
| `z` | Infrared filter magnitude |
| `redshift` | How much the spectral lines are shifted toward red (distance proxy) |
| `class` | Target label: **STAR**, **GALAXY**, or **QSO** (quasar) |

The five magnitude values (`u`, `g`, `r`, `i`, `z`) are analogous to measuring brightness through five coloured filters. Together with redshift, they form the spectral fingerprint the network will learn to classify.

> **Why this problem?** In the reading material you saw how neural networks excel at tasks where the mapping from inputs to outputs is too complex for hand-written rules. This is exactly that kind of problem — the spectral differences between a star, a galaxy, and a quasar are subtle and multidimensional, making it a perfect case for a neural network.

In [ ]:
# Generate simulated SDSS-style stellar spectra dataset
np.random.seed(42)
n_samples = 3000
n_per_class = n_samples // 3

def make_stars(n):
    """Simulate stellar photometry — stars are bright, nearby, low redshift."""
    u = np.random.normal(19.5, 0.8, n)
    g = np.random.normal(18.8, 0.7, n)
    r = np.random.normal(18.3, 0.6, n)
    i = np.random.normal(18.0, 0.6, n)
    z = np.random.normal(17.8, 0.5, n)
    redshift = np.abs(np.random.normal(0.0002, 0.0001, n))
    return np.column_stack([u, g, r, i, z, redshift])

def make_galaxies(n):
    """Simulate galaxy photometry — dimmer, moderate redshift."""
    u = np.random.normal(21.5, 1.0, n)
    g = np.random.normal(20.5, 0.9, n)
    r = np.random.normal(19.8, 0.8, n)
    i = np.random.normal(19.2, 0.8, n)
    z = np.random.normal(18.9, 0.7, n)
    redshift = np.abs(np.random.normal(0.08, 0.04, n))
    return np.column_stack([u, g, r, i, z, redshift])

def make_qsos(n):
    """Simulate quasar photometry — very high redshift, distinctive colours."""
    u = np.random.normal(20.0, 1.2, n)
    g = np.random.normal(19.8, 1.1, n)
    r = np.random.normal(19.6, 1.0, n)
    i = np.random.normal(19.3, 1.0, n)
    z = np.random.normal(19.0, 0.9, n)
    redshift = np.abs(np.random.normal(1.5, 0.8, n))
    return np.column_stack([u, g, r, i, z, redshift])

X_stars    = make_stars(n_per_class)
X_galaxies = make_galaxies(n_per_class)
X_qsos     = make_qsos(n_per_class)

X_raw = np.vstack([X_stars, X_galaxies, X_qsos])
y_raw = np.array(['STAR'] * n_per_class + ['GALAXY'] * n_per_class + ['QSO'] * n_per_class)

feature_names = ['u', 'g', 'r', 'i', 'z', 'redshift']
df = pd.DataFrame(X_raw, columns=feature_names)
df['class'] = y_raw

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['class'].value_counts())
print(f'\nFirst 5 rows:')
df.head()

### 1.2 — Visualise the Feature Distributions

Before building any model, it is always worth looking at the data. The plots below show how each feature is distributed across the three classes. This helps us understand:

- Which features are most discriminative (most different between classes)
- Whether the classes overlap significantly (which will make classification harder)
- Whether any preprocessing is obviously needed

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flat
colors = {'STAR': '#2196F3', 'GALAXY': '#4CAF50', 'QSO': '#FF5722'}

for i, feat in enumerate(feature_names):
    for cls in ['STAR', 'GALAXY', 'QSO']:
        subset = df[df['class'] == cls][feat]
        axes[i].hist(subset, bins=40, alpha=0.6, label=cls, color=colors[cls])
    axes[i].set_xlabel(feat, fontsize=11)
    axes[i].set_ylabel('Count', fontsize=11)
    axes[i].set_title(f'Distribution of {feat}', fontsize=12)
    axes[i].legend(fontsize=9)

plt.suptitle('Feature Distributions by Object Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n📊 Observation: Which feature separates the classes most clearly?")
print("   Notice how 'redshift' is almost perfectly discriminative for QSOs.")
print("   The magnitude features (u, g, r, i, z) show more overlap — the network")
print("   will need to learn to use these together, not individually.")

---
## Step 2 — Preprocess the Data

Before feeding data into a neural network, we need to do three things:

1. **Encode the labels** — neural networks work with numbers, not strings. We convert STAR/GALAXY/QSO into 0/1/2.
2. **Standardise the features** — different features have very different scales (magnitudes are around 18–22; redshift is 0–3). Neural networks train more stably when all features are on the same scale. We use `StandardScaler` to give each feature a mean of 0 and standard deviation of 1.
3. **Split into training and test sets** — we keep 20% of the data as a held-out test set. The model never trains on these — they are for final evaluation only.

> **Why standardise?** From the reading material, you know that neurons compute a weighted sum of their inputs. If one feature has values in the range [0, 3] and another has values in the range [17, 23], the weights for the large-valued feature will be pushed toward very small values to compensate. Standardisation removes this asymmetry and allows gradient descent to converge much more reliably.

In [ ]:
# Encode labels: GALAXY=0, QSO=1, STAR=2 (alphabetical by default)
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
class_names = le.classes_
n_classes = len(class_names)

print(f'Label encoding: {dict(zip(class_names, le.transform(class_names)))}')
print(f'Number of classes: {n_classes}')

# Train / test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Standardise features — fit on training set only, apply to both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # use training mean/std — no data leakage

print(f'\nTraining set: {X_train_scaled.shape[0]} samples')
print(f'Test set:     {X_test_scaled.shape[0]} samples')
print(f'Features:     {X_train_scaled.shape[1]}')
print(f'\nAfter scaling — training set feature statistics:')
print(f'  Mean: {X_train_scaled.mean(axis=0).round(3)}')
print(f'  Std:  {X_train_scaled.std(axis=0).round(3)}')

---
## Step 3 — Build the MLP

### 3.1 — Architecture Design

We will build a **Multi-Layer Perceptron (MLP)** — the simplest form of deep neural network, consisting entirely of fully connected (dense) layers. This is the architecture described in the reading material.

Our network design:

```
Input (6 features)
    ↓
Dense(128, ReLU)         ← first hidden layer
Dropout(0.3)             ← drop 30% of neurons randomly during training
    ↓
Dense(64, ReLU)          ← second hidden layer
Dropout(0.3)
    ↓
Dense(32, ReLU)          ← third hidden layer
    ↓
Dense(3, Softmax)        ← output: probability for each of 3 classes
```

**Design decisions explained:**

- **ReLU activations** on hidden layers — as you learned in the reading, ReLU is the standard choice for hidden layers. It introduces non-linearity cheaply and avoids vanishing gradients.
- **Softmax on the output layer** — for multi-class classification (3 classes), softmax converts the raw output values into probabilities that sum to 1. The predicted class is whichever has the highest probability.
- **Dropout** — during each training step, dropout randomly sets 30% of a layer's neurons to zero. This prevents the network from relying too heavily on any individual neuron, which reduces overfitting.
- **Decreasing layer sizes** (128→64→32) — this funnel shape forces the network to progressively compress its representation, encouraging it to learn the most informative features.

In [ ]:
def build_mlp(input_dim, n_classes, dropout_rate=0.3):
    """Build a fully connected MLP for multi-class classification."""
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),

        # Hidden layer 1
        layers.Dense(128, activation='relu', name='hidden_1'),
        layers.Dropout(dropout_rate, name='dropout_1'),

        # Hidden layer 2
        layers.Dense(64, activation='relu', name='hidden_2'),
        layers.Dropout(dropout_rate, name='dropout_2'),

        # Hidden layer 3
        layers.Dense(32, activation='relu', name='hidden_3'),

        # Output layer — softmax for multi-class probability
        layers.Dense(n_classes, activation='softmax', name='output')
    ], name='stellar_mlp')
    return model

model = build_mlp(input_dim=X_train_scaled.shape[1], n_classes=n_classes)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### 3.2 — Count the Parameters

The model summary above shows the number of trainable parameters in each layer. Let's verify the calculation for the first hidden layer:

- Input: 6 features
- Hidden layer 1: 128 neurons
- Parameters = (6 inputs × 128 neurons) + 128 biases = **896 parameters**

Each of those 896 numbers is a weight or bias that gradient descent will adjust during training. This is exactly the learning process described in the reading material.

In [ ]:
# Verify parameter count for first hidden layer
input_dim = X_train_scaled.shape[1]
neurons_l1 = 128
params_l1 = (input_dim * neurons_l1) + neurons_l1  # weights + biases

print(f'Layer 1 parameter count: ({input_dim} inputs × {neurons_l1} neurons) + {neurons_l1} biases = {params_l1}')
print(f'\nTotal trainable parameters: {model.count_params():,}')
print(f'\nThis is the number of values gradient descent will optimise during training.')

---
## Step 4 — Train the Model

### 4.1 — Early Stopping

We will train with **early stopping** — a technique that monitors the validation loss at the end of each epoch and stops training automatically if it has not improved for a specified number of epochs (the `patience`). This prevents overfitting by stopping before the model starts memorising the training data.

> **Connection to the reading:** The reading explained that overfitting occurs when a model learns the specific noise of the training data rather than the underlying patterns. Early stopping is one of the most practical defences against this — it is essentially letting the validation loss tell you when to stop.

In [ ]:
# Early stopping: stop if validation loss doesn't improve for 10 epochs
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,   # restore the weights from the best epoch
    verbose=1
)

print('Training the MLP on stellar spectra...')
print('Early stopping will kick in if validation loss plateaus for 10 epochs.\n')

history = model.fit(
    X_train_scaled, y_train,
    epochs=150,
    batch_size=64,
    validation_split=0.2,         # 20% of training set used for validation
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Training complete. Stopped at epoch {len(history.history["loss"])}.')

### 4.2 — Plot the Learning Curves

The learning curves show how the model's **loss** and **accuracy** changed over training epochs — separately for the training set and the validation set.

**What to look for:**
- Both curves should decrease (loss) or increase (accuracy) over time
- A **small, stable gap** between training and validation is healthy
- A **large and growing gap** (training improves but validation plateaus or worsens) indicates overfitting
- Curves that oscillate wildly may indicate a learning rate that is too high

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history.history['loss'],     label='Training loss',   color='steelblue', linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation loss', color='tomato',    linewidth=2, linestyle='--')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Sparse Categorical Cross-Entropy Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Accuracy curves
ax2.plot(history.history['accuracy'],     label='Training accuracy',   color='steelblue', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Validation accuracy', color='tomato',    linewidth=2, linestyle='--')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.suptitle('MLP Learning Curves — Stellar Spectra Classification', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
print(f'Final training accuracy:   {final_train_acc:.4f}')
print(f'Final validation accuracy: {final_val_acc:.4f}')
print(f'Gap (train - val):         {final_train_acc - final_val_acc:.4f}')

---
## Step 5 — Evaluate on the Test Set

The test set has been held out throughout training — the model has never seen these examples. This gives us an unbiased estimate of how well the model will perform on genuinely new data.

### 5.1 — Overall Test Accuracy

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}  ({test_acc*100:.1f}%)')

y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

### 5.2 — Confusion Matrix

A confusion matrix shows exactly where the model makes mistakes. Each row is the **true class**; each column is the **predicted class**. Numbers on the diagonal are correct predictions; off-diagonal entries are errors.

This tells you much more than overall accuracy — for example, you might find that the model confuses Stars with Galaxies more often than Stars with QSOs.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 14}
)
plt.xlabel('Predicted Class', fontsize=13)
plt.ylabel('True Class', fontsize=13)
plt.title('Confusion Matrix — Stellar Spectra Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_names))

### 5.3 — Per-Class Confidence

For each test example, the model outputs a probability for each class (via the softmax output layer). Let's look at the average confidence the model assigns to the correct class, broken down by object type.

In [ ]:
# Average predicted probability for the correct class, per true class
for cls_idx, cls_name in enumerate(class_names):
    mask = y_test == cls_idx
    correct_probs = y_pred_probs[mask, cls_idx]
    print(f'{cls_name:10s}  — mean confidence on correct class: {correct_probs.mean():.3f}  '
          f'(min: {correct_probs.min():.3f}, max: {correct_probs.max():.3f})')

---
## Step 6 — Visualise the Learned Weights

### 6.1 — What Are We Looking At?

The weight matrix of the **first hidden layer** has shape `(6 inputs × 128 neurons)`. Each column represents one neuron. The values in each column are the weights connecting the 6 input features to that neuron — large positive weights mean the neuron is strongly activated by that feature; large negative weights mean the neuron is suppressed by it.

Visualising this matrix gives us a window into what the network found important in the first transformation step — which input features it emphasised and which it de-emphasised.

> **Why does this matter?** In the reading, you learned that early layers detect simple patterns, and later layers build on these to detect more complex ones. This visualisation lets you see the very first layer of that hierarchy — what raw combinations of spectral features the network learned to respond to.

In [ ]:
# Extract weights from the first hidden layer
weights_layer1, biases_layer1 = model.get_layer('hidden_1').get_weights()

print(f'Weight matrix shape: {weights_layer1.shape}  → ({len(feature_names)} inputs × {weights_layer1.shape[1]} neurons)')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of all weights
im = axes[0].imshow(weights_layer1, aspect='auto', cmap='RdBu_r',
                    vmin=-weights_layer1.abs().max() if hasattr(weights_layer1, 'abs') else -np.abs(weights_layer1).max(),
                    vmax=np.abs(weights_layer1).max())
axes[0].set_xlabel('Neuron Index (Hidden Layer 1)', fontsize=12)
axes[0].set_ylabel('Input Feature', fontsize=12)
axes[0].set_yticks(range(len(feature_names)))
axes[0].set_yticklabels(feature_names, fontsize=11)
axes[0].set_title('Weight Matrix: 6 Inputs × 128 Neurons\n(Red = positive, Blue = negative)', fontsize=12)
plt.colorbar(im, ax=axes[0], label='Weight Value')

# Average absolute weight per feature — importance proxy
mean_abs_weights = np.abs(weights_layer1).mean(axis=1)
bars = axes[1].barh(feature_names, mean_abs_weights,
                    color=['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800','#00BCD4'])
axes[1].set_xlabel('Mean Absolute Weight', fontsize=12)
axes[1].set_title('Average Feature Importance\n(Mean |weight| across all 128 neurons)', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')
for bar, val in zip(bars, mean_abs_weights):
    axes[1].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)

plt.suptitle('Learned Weights — First Hidden Layer', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

most_important = feature_names[np.argmax(mean_abs_weights)]
print(f'\nFeature with largest average weight: {most_important}')
print(f'This suggests the network found {most_important} most informative in the first layer.')

---
## Step 7 — Guided Reflection and Exercises

Answer the questions below by replacing the placeholder text with your observations. These are thinking prompts, not right-or-wrong questions.

---

### ❓ Question 1
**Looking at the feature distribution plots in Step 1: which feature separates the three classes most clearly, and which shows the most overlap? Does the weight visualisation in Step 6 agree with your intuition?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 2
**Look at the learning curves from Step 4. Is there a visible gap between training and validation accuracy? Does this suggest overfitting? What did early stopping do?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 3
**Look at the confusion matrix. Which two classes does the model most commonly confuse? Why might those classes be harder to separate given their feature distributions?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 4 — Optional experiment
**Remove the Dropout layers from the model (set `dropout_rate=0.0`) and retrain. How do the training and validation accuracy curves change? Does the model overfit more?**

In [ ]:
# Optional: retrain without dropout and compare
# Change dropout_rate to 0.0 below and run this cell

model_no_dropout = build_mlp(input_dim=X_train_scaled.shape[1], n_classes=n_classes, dropout_rate=0.0)
model_no_dropout.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_no_drop = model_no_dropout.fit(
    X_train_scaled, y_train,
    epochs=150, batch_size=64, validation_split=0.2,
    callbacks=[early_stop], verbose=0
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['val_loss'],          label='With dropout',    color='steelblue', linewidth=2)
ax1.plot(history_no_drop.history['val_loss'],  label='Without dropout', color='tomato',    linewidth=2, linestyle='--')
ax1.set_title('Validation Loss: Dropout vs No Dropout')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['accuracy'],              label='Train (with dropout)',    color='steelblue', linewidth=2)
ax2.plot(history.history['val_accuracy'],          label='Val (with dropout)',      color='steelblue', linewidth=2, linestyle='--')
ax2.plot(history_no_drop.history['accuracy'],      label='Train (no dropout)',      color='tomato',    linewidth=2)
ax2.plot(history_no_drop.history['val_accuracy'],  label='Val (no dropout)',        color='tomato',    linewidth=2, linestyle='--')
ax2.set_title('Train vs Val Accuracy: Dropout Comparison')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## ✅ Notebook 1 Complete

You have:

| ✅ | Achievement |
|----|-------------|
| ✅ | Loaded and explored a simulated SDSS stellar spectra dataset |
| ✅ | Preprocessed data: standardisation, label encoding, train/test split |
| ✅ | Built a fully connected MLP with dropout in TensorFlow/Keras |
| ✅ | Trained with early stopping and interpreted learning curves |
| ✅ | Evaluated using a confusion matrix and per-class classification report |
| ✅ | Visualised and interpreted learned weights from the first hidden layer |

---

### ➡️ Next Step: Notebook 2

**Before opening Notebook 2**, read the Lesson 2 reading pages on **Overfitting and Generalisation** and **Hyperparameter Tuning**. Notebook 2 takes the concepts from those pages — regularisation, depth, learning rate, batch size — and lets you explore their effects interactively through a series of controlled experiments.